# Figure S17

Compares groundwater pumping among the three response classes.


In [1]:
from pathlib import Path
import warnings

import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings('ignore', category=RuntimeWarning)


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'outputs' / 'RECON_MAIN_2011_2023').exists():
            return candidate
    raise FileNotFoundError('Could not find outputs/RECON_MAIN_2011_2023.')


def display_path(path: Path) -> str:
    return str(path.resolve().relative_to(ROOT))


ROOT = find_repo_root()
RECON = ROOT / 'outputs' / 'RECON_MAIN_2011_2023'
LABELS_PATH = RECON / 'metrics' / 'clustering' / 'cluster_labels.csv'
DYNAMIC_PATH = ROOT / 'data' / 'train_val_test_inputs' / 'GNN_spacetime' / 'H6' / 'dynamic_monthly_cache.pt'
OUT_DIR = ROOT / 'outputs' / 'figures' / 'FigS17'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PUMPING_FIG_PATH = OUT_DIR / 'FigS17_cd_pumping.png'

CLASS_ORDER = ['Fast recovery', 'Slow recovery', 'Buffered']
CLASS_COLORS = {
    'Fast recovery': '#2D5FB8',
    'Slow recovery': '#C44E72',
    'Buffered': '#13A8A2',
}
EXPORT_DPI = 600

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 9.0,
    'axes.labelsize': 9.5,
    'axes.titlesize': 10.0,
    'xtick.labelsize': 8.3,
    'ytick.labelsize': 8.3,
    'legend.fontsize': 8.2,
    'axes.linewidth': 0.75,
    'axes.unicode_minus': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.dpi': EXPORT_DPI,
    'savefig.bbox': 'tight',
})

In [2]:
labels = pd.read_csv(LABELS_PATH)
required = {
    'grid_id', 'row', 'col', 'x', 'y', 'valid_for_clustering', 'response_class',
}
missing = required.difference(labels.columns)
if missing:
    raise KeyError(f'Missing required label columns: {sorted(missing)}')

labels['valid_for_clustering'] = labels['valid_for_clustering'].astype(bool)
valid = labels['valid_for_clustering'].to_numpy() & labels['response_class'].notna().to_numpy()
response_class = labels['response_class'].to_numpy()

PUMPING_MONTHS = np.arange(5, 10)
cache = torch.load(DYNAMIC_PATH, map_location='cpu', weights_only=False)
if not np.array_equal(np.asarray(cache['grid_ids']), labels['grid_id'].to_numpy()):
    raise ValueError('Pumping cache and class labels are not aligned by grid_id.')

feature_index = {name: index for index, name in enumerate(cache['feature_names'])}
if 'monthly_pumping' not in feature_index:
    raise KeyError('Missing dynamic forcing feature: monthly_pumping')

dynamic_data = np.asarray(cache['data'], dtype=np.float32)
pumping_m3 = dynamic_data[:, :, feature_index['monthly_pumping']]
month_labels = np.asarray(cache['month_labels']).astype(str)
years = np.asarray([int(label[:4]) for label in month_labels])
months = np.asarray([int(label[5:7]) for label in month_labels])

drought_2012 = (years == 2012) & (months >= 5) & (months <= 10)
if drought_2012.sum() != 6:
    raise ValueError('The 2012 May–October pumping window must contain six months.')
pumping_2012_m3 = np.nansum(pumping_m3[:, drought_2012], axis=1)

active_2012_months = (years == 2012) & np.isin(months, PUMPING_MONTHS)
pumping_2012_peak_m3 = np.nanmax(pumping_m3[:, active_2012_months], axis=1)
pumping_2012_active_grid = np.any(
    pumping_m3[:, active_2012_months] > 1e-6, axis=1
)
growing_season_totals_m3 = np.stack([
    np.nansum(
        pumping_m3[:, (years == year) & np.isin(months, PUMPING_MONTHS)],
        axis=1,
    )
    for year in np.unique(years)
], axis=1)
pumping_longterm_seasonal_mean_m3 = np.nanmean(
    growing_season_totals_m3, axis=1
)

pumping_2012_monthly_median = {}
pumping_signature_raw = {}
for class_name in CLASS_ORDER:
    class_cells = valid & (response_class == class_name)
    medians = []
    for month in PUMPING_MONTHS:
        month_index = np.where((years == 2012) & (months == month))[0]
        if month_index.size != 1:
            raise ValueError(f'Expected one 2012-{month:02d} pumping layer.')
        values = pumping_m3[class_cells, month_index[0]].astype(float)
        values = values[np.isfinite(values)]
        medians.append(float(np.median(values)))
    pumping_2012_monthly_median[class_name] = np.asarray(medians, dtype=float)
    pumping_signature_raw[class_name] = np.asarray([
        np.nanmedian(pumping_2012_m3[class_cells]),
        np.nanmedian(pumping_2012_peak_m3[class_cells]),
        np.nanmedian(pumping_longterm_seasonal_mean_m3[class_cells]),
        100.0 * pumping_2012_active_grid[class_cells].mean(),
    ], dtype=float)

PUMPING_SIGNATURE_LABELS = [
    '2012 seasonal\ntotal', '2012 peak\nmonth',
    '2011–2023\nseasonal mean', 'Irrigated\ngrid fraction',
]
fast_signature = pumping_signature_raw['Fast recovery']
pumping_signature_relative = {
    class_name: pumping_signature_raw[class_name] / fast_signature
    for class_name in CLASS_ORDER
}

print(f'Valid classified cells: {valid.sum():,}')
print('2012 pumping window: May–October; units: original pumping volume (m³)')

Valid classified cells: 87,647
2012 pumping window: May–October; units: original pumping volume (m³)


In [3]:
def class_mask(class_name: str) -> np.ndarray:
    return valid & (response_class == class_name)


def style_axis(ax: plt.Axes, grid_axis: str | None = None) -> None:
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.75)
        spine.set_color('#333333')
    ax.tick_params(length=3.0, width=0.7, direction='out')
    if grid_axis is not None:
        ax.grid(axis=grid_axis, color='#E5E5E5', linewidth=0.45, zorder=0)
        ax.set_axisbelow(True)


fig_pumping = plt.figure(figsize=(10.2, 4.5), facecolor='white')
pumping_grid = fig_pumping.add_gridspec(
    1, 2, width_ratios=[1.15, 1.55], wspace=0.42,
)
ax_c = fig_pumping.add_subplot(pumping_grid[0, 0])
ax_d = fig_pumping.add_subplot(pumping_grid[0, 1])

month_positions = np.arange(len(PUMPING_MONTHS), dtype=float)
month_labels_short = ['May', 'Jun', 'Jul', 'Aug', 'Sep']
bar_width = 0.24
for class_idx, class_name in enumerate(CLASS_ORDER):
    offset = (class_idx - 1) * bar_width
    ax_c.bar(
        month_positions + offset, pumping_2012_monthly_median[class_name],
        width=bar_width, color=CLASS_COLORS[class_name], alpha=0.86,
        edgecolor='white', linewidth=0.50, label=class_name,
    )
ax_c.set_xlim(-0.55, len(PUMPING_MONTHS) - 0.45)
ax_c.set_ylim(0, 55_000)
ax_c.set_xticks(month_positions)
ax_c.set_xticklabels(month_labels_short, fontsize=7.7)
ax_c.set_yticks([0, 10_000, 20_000, 30_000, 40_000, 50_000])
ax_c.set_yticklabels(['0', '10,000', '20,000', '30,000', '40,000', '50,000'])
ax_c.set_xlabel('Month')
ax_c.set_ylabel('Median monthly pumping (m³)')
ax_c.legend(frameon=False, loc='upper left', handlelength=1.7, fontsize=7.8)
style_axis(ax_c, 'y')

signature_x = np.arange(len(PUMPING_SIGNATURE_LABELS), dtype=float)
plot_order = ['Fast recovery', 'Buffered', 'Slow recovery']
for class_name in plot_order:
    values = pumping_signature_relative[class_name]
    ax_d.plot(
        signature_x, values, color=CLASS_COLORS[class_name], linewidth=2.0,
        marker='o', markersize=5.2, markeredgecolor='white', markeredgewidth=0.55,
    )
    if class_name != 'Fast recovery':
        offset = 0.035 if class_name == 'Slow recovery' else -0.045
        va = 'bottom' if offset > 0 else 'top'
        for x_pos, value in zip(signature_x, values):
            ax_d.text(x_pos, value + offset, f'{value:.2f}×', ha='center', va=va,
                      fontsize=7.2, color=CLASS_COLORS[class_name], fontweight='bold')
ax_d.axhline(1.0, color='#777777', linewidth=0.75, linestyle='--', zorder=0)
ax_d.set_xlim(-0.20, len(PUMPING_SIGNATURE_LABELS) - 0.80)
ax_d.set_ylim(0.90, 1.70)
ax_d.set_xticks(signature_x)
ax_d.set_xticklabels(PUMPING_SIGNATURE_LABELS, fontsize=7.5)
ax_d.set_yticks([1.0, 1.2, 1.4, 1.6])
ax_d.set_ylabel('Relative to Fast recovery')
ax_d.text(0.02, 0.04, 'Fast recovery = 1', transform=ax_d.transAxes,
          fontsize=7.2, color='#555555', ha='left', va='bottom')
style_axis(ax_d, 'y')

fig_pumping.savefig(PUMPING_FIG_PATH, dpi=EXPORT_DPI, facecolor='white')
plt.show()

print('Saved:', display_path(PUMPING_FIG_PATH))
print('Class summary:')
for class_name in CLASS_ORDER:
    mask = class_mask(class_name)
    seasonal_total, peak_month, longterm_mean, active_fraction = pumping_signature_raw[class_name]
    print(
        f'  {class_name}: n={mask.sum():,}, '
        f'2012 seasonal total={seasonal_total:,.0f} m³, '
        f'peak month={peak_month:,.0f} m³, '
        f'long-term seasonal mean={longterm_mean:,.0f} m³, '
        f'active grid fraction={active_fraction:.1f}%'
    )

Saved: outputs\figures\FigS17\FigS17_cd_pumping.png
Class summary:
  Fast recovery: n=33,155, 2012 seasonal total=95,258 m³, peak month=32,649 m³, long-term seasonal mean=83,654 m³, active grid fraction=68.3%
  Slow recovery: n=30,854, 2012 seasonal total=150,896 m³, peak month=51,066 m³, long-term seasonal mean=126,615 m³, active grid fraction=78.9%
  Buffered: n=23,638, 2012 seasonal total=112,102 m³, peak month=38,336 m³, long-term seasonal mean=115,009 m³, active grid fraction=75.2%


C:\Users\10211\AppData\Local\Temp\ipykernel_8736\2824028430.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
